# Live-Action Drama & 1:1 MV Studio V3 (Krea-2 Identity V2 + MiniMax-H3 FP8 + SageAttention)

Notebook sản xuất tinh gọn chuẩn **V3** (đã kiểm chứng Full MV 4:38 / 143 phân cảnh trên Colab L4 24GB, hỗ trợ tự động cả **G4 96GB**, **L4 24GB**, **A100 80GB** và **H100 80GB**).

### Bảng Công Thức Tối Ưu Theo GPU
| GPU Colab | Giá thuê | Khuyến nghị dùng | Tốc độ thực tế (`B2 @ 960×544`) |
|---|---:|---|---|
| **G4 (RTX PRO 6000 96 GB)** | `6.7–8.7 CU/h` | **#1 Vô địch P/P** (FP8 + FP4 native, 0 offload) | `~28–35 phút` / Full MV (`~3.5–4.6 CU`) |
| **L4 (24 GB, 52 GB RAM)** | `1.94 CU/h` | **#2 Kinh tế & dễ bắt nhất** (Warm-RAM `B2`, treo `/goal`) | `50–144s/clip` • `~2h46` / Full MV (`~5.2 CU`) |
| **A100-80GB** | `8.88 CU/h` | Dùng khi cần xong gấp `<1h` hoặc chạy `736p` 20-step | `~45–55 phút` / Full MV (`~7.5 CU`) |

### Quy Tắc Sống Còn (Tránh OOM & Lỗi Hình Ảnh)
1. **Dùng `FP8_scaled` (`B2` = EasyCache `0.10`, fallback `B1` = EasyCache OFF):** Không dùng `W6A8` 6-bit (gây giật chuyển frame).
2. **Trên L4 (52 GB RAM):** Không chạy `RealESRGAN_x4plus` chung ComfyUI với MiniMax-H3 (dễ OOM ở 52.4 GB) — luôn xuất native `960×544` và upscale 1080p ở máy local bằng FFmpeg (`Lanczos + CAS 0.28 + 35mm grain`, `0 CU`).
3. **Bắt buộc gọi `/free` giữa Krea-2 (`19 GB`) và MiniMax-H3 (`44 GB`):** Cell 4 đã tự xử lý hoàn toàn.
4. **Khóa tỷ lệ nhân vật 1:1:** Krea-2 hiệu chỉnh `ref_boost/ref_boost_a` theo cự ly (`WS: 1.85/1.18` → `CU: 2.75/1.00`) + ưu tiên đơn khung **`I2VA`** để triệt sạch hiện tượng phóng to/dissolve giữa clip.
5. **Chống cháy CU:** Chạy **Cell 7** ngay khi tải xong sản phẩm để gọi `runtime.unassign()` ngắt GPU tức thì.

---
**Thứ tự chạy:** `Cell 1` (Cài & mở ComfyUI + Tunnel) → *(tuỳ chọn `Cell 2` mount Drive)* → `Cell 3` (Krea-2 Swap) → `Cell 4` (MiniMax-H3 Batch `B2`) hoặc `Cell 5` (Queue JSON `Ref2VA`) → `Cell 6` (Theo dõi log/GPU) → `Cell 7` (Tắt & nhả GPU).


In [ ]:
# CELL 1: 1-Click Auto-Setup (ComfyUI + Krea-2 Identity V2 + MiniMax-H3 FP8_scaled + SageAttention + Cloudflared)
# Zero Google Drive OAuth needed! Runs detached in ~3.5-4.0 min -> check progress in Cell 6.
import subprocess, sys, time
from pathlib import Path

INSTALL_REF2VA = False   # Set True only if running Ref2VA dialogue/multi-ref mode (+21 GB)
INSTALL_REALISM = False  # Set True to also fetch fal/MiniMax-H3-Realism-People-LoRA (+131 MB)

SETUP_SCRIPT = Path("/tmp/setup_studio_v3.py")
SETUP_SCRIPT.write_text(f"""import os, re, subprocess, sys, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

INSTALL_REF2VA = {INSTALL_REF2VA}
INSTALL_REALISM = {INSTALL_REALISM}
COMFY_DIR = Path("/content/ComfyUI")
MODELS = COMFY_DIR / "models"
WORK_DIR = Path("/content/mv_studio")
for d in ["inputs", "swapped", "videos", "outputs"]:
    (WORK_DIR / d).mkdir(parents=True, exist_ok=True)

t0 = time.time()
print("=== [1/5] GPU CHECK & COMFYUI CORE + KREA2EDIT ===", flush=True)
print(subprocess.check_output("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && free -h", shell=True, text=True), flush=True)
if not COMFY_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "av", "imageio-ffmpeg"], check=True)

node_dir = COMFY_DIR / "custom_nodes" / "comfyui-krea2edit"
if not node_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lbouaraba/comfyui-krea2edit.git", str(node_dir)], check=True)
    req = node_dir / "requirements.txt"
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

print("=== [2/5] INSTALLING SAGEATTENTION ===", flush=True)
sage_ok = False
try:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sageattention"], capture_output=True, text=True, timeout=180)
    if r.returncode == 0:
        r2 = subprocess.run([sys.executable, "-c", "import sageattention; print('SAGE_OK:', sageattention.__file__)"], capture_output=True, text=True)
        print(r2.stdout.strip() or r2.stderr.strip(), flush=True)
        sage_ok = (r2.returncode == 0)
except Exception as e:
    print("SageAttention fallback to SDPA:", e, flush=True)

print("=== [3/5] PARALLEL MODEL DOWNLOAD (Zero-Duplicate Symlinks) ===", flush=True)
from huggingface_hub import hf_hub_download

jobs = [
    # Krea-2 Turbo + Identity Edit V2
    ("Comfy-Org/Krea-2", "diffusion_models/krea2_turbo_int8_convrot.safetensors", "diffusion_models"),
    ("Comfy-Org/Krea-2", "text_encoders/qwen3vl_4b_fp8_scaled.safetensors", "text_encoders"),
    ("Comfy-Org/Krea-2", "vae/qwen_image_vae.safetensors", "vae"),
    ("conradlocke/krea2-identity-edit", "krea2_identity_edit_v2_r64.safetensors", "loras"),
    # MiniMax-H3 Core + Proven Smooth FP8_scaled + Kijai Pruned 8-Step v1.2
    ("Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
    ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors", "diffusion_models"),
    ("Kijai/MiniMax-H3_comfy", "loras/minimax_h3_fl2v_turbo_8step_v1.2_768p_pruned_avg_rank_24_bf16.safetensors", "loras"),
]
if INSTALL_REF2VA:
    jobs += [
        ("Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", "diffusion_models"),
        ("lightx2v/Minimax-h3-Turbo", "minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors", "loras"),
    ]
if INSTALL_REALISM:
    jobs.append(("fal/MiniMax-H3-Realism-People-LoRA", "h3-realism-people-t2v-i2v-r2v.safetensors", "loras"))

def dl(item):
    repo_id, repo_file, rel_dir = item
    tdir = MODELS / rel_dir
    tdir.mkdir(parents=True, exist_ok=True)
    dst = tdir / Path(repo_file).name
    if dst.exists() and dst.stat().st_size > 10_000:
        return f"SKIP: {{dst.name}}"
    cached = hf_hub_download(repo_id=repo_id, filename=repo_file)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(cached, dst)
    return f"OK ({{Path(cached).stat().st_size/1e6:.1f}} MB): {{dst.name}}"

with ThreadPoolExecutor(max_workers=4) as ex:
    for msg in ex.map(dl, jobs):
        print(" ", msg, flush=True)

print("=== [4/5] LAUNCHING COMFYUI (Warm-RAM + SageAttention) ===", flush=True)
subprocess.run(["pkill", "-9", "-f", "main.py --listen"], check=False)
time.sleep(2)
cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"]
if sage_ok:
    cmd.append("--use-sage-attention")
clog = Path("/tmp/comfy_server.log").open("w", encoding="utf-8")
subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=clog, stderr=subprocess.STDOUT, start_new_session=True)

def alive():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

for _ in range(90):
    if alive():
        break
    time.sleep(1)
assert alive(), f"ComfyUI failed to start!\\n{{Path('/tmp/comfy_server.log').read_text(errors='replace')[-1200:]}}"
print(f"COMFYUI_LIVE_OK (sage={{sage_ok}}) in {{time.time()-t0:.1f}}s!", flush=True)

print("=== [5/5] STARTING CLOUDFLARED TUNNEL (Zero-OAuth Transfer) ===", flush=True)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)
cf_bin = Path("/usr/local/bin/cloudflared")
if not cf_bin.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", str(cf_bin))
    os.chmod(cf_bin, 0o755)
cf_log = Path("/tmp/cf_tunnel.log").open("w", encoding="utf-8")
subprocess.Popen([str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8188"], stdout=cf_log, stderr=subprocess.STDOUT, start_new_session=True)
url = ""
for _ in range(30):
    m = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", Path("/tmp/cf_tunnel.log").read_text(errors="replace"))
    if m:
        url = m.group(0)
        break
    time.sleep(1)
Path("/tmp/tunnel_url.txt").write_text(url, encoding="utf-8")
print("TUNNEL_URL:", url, flush=True)
print("=== SETUP_STUDIO_V3_DONE ===", flush=True)
""", encoding="utf-8")

log_fh = Path("/tmp/setup_studio_v3.log").open("w", encoding="utf-8")
subprocess.Popen([sys.executable, "-u", str(SETUP_SCRIPT)], stdout=log_fh, stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(5)
print("DETACHED SETUP V3 LAUNCHED (~3.5-4.0 min). Run Cell 6 to monitor progress:")
print(Path("/tmp/setup_studio_v3.log").read_text(errors="replace"))


In [ ]:
# CELL 2 (OPTIONAL): Mount Google Drive for Direct Sync
# Skip this cell if using Zero-OAuth HTTP transfer via Cloudflared (/upload/image & /view).
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/DramaStudio")
for sub in ["inputs", "projects", "outputs/videos"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Google Drive mounted cleanly at:", DRIVE_ROOT)


In [ ]:
# CELL 3: Stage 1 — Reusable Scale-Locked Krea-2 Identity Edit V2 Batch Runner
# Reads /content/mv_studio/jobs.json (or PNG tEXt payload /content/ComfyUI/input/jobs_payload.png)
# Calibrates ref_boost & ref_boost_a by shot scale (WS: 1.85/1.18 -> CU: 2.75/1.00) to prevent actor bloating.
import json, shutil, time, urllib.parse, urllib.request, uuid
from pathlib import Path
from PIL import Image, ImageDraw

SERVER = "http://127.0.0.1:8188"
COMFY_IN = Path("/content/ComfyUI/input")
COMFY_OUT = Path("/content/ComfyUI/output")
SWAP_DIR = Path("/content/mv_studio/swapped")
SWAP_DIR.mkdir(parents=True, exist_ok=True)

REF_BOOST   = {"CU": 2.75, "MCU": 2.55, "MS": 2.30, "WS": 1.85}
REF_BOOST_A = {"CU": 1.00, "MCU": 1.00, "MS": 1.08, "WS": 1.18}

DEFAULT_MALE_SYS = (
    "Replace ONLY the man's face and hair with the exact young East Asian man from the second image "
    "(tousled dark black bangs, sharp jawline, cool eyes, tiny beauty mark below the right corner of the lower lip). "
    "NEVER copy his suit/shirt — strictly preserve the EXACT outfit, figure height, pose, and background of the first image."
)
DEFAULT_FEMALE_SYS = (
    "Replace ONLY the woman's face and hair with the exact young East Asian woman from the second image "
    "(sleek straight dark black hair, small silver side hairpin, porcelain V-line face, luminous almond eyes). "
    "NEVER copy her blazer/uniform — strictly preserve the EXACT outfit, figure height, pose, and background of the first image."
)

def _http_json(url, payload=None, timeout=60):
    req = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST") if payload else urllib.request.Request(url)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

def build_krea2_swap(scene_img, actor_img, prompt, sys_prompt, out_stem, seed, scale="MCU", rb=None, rba=None, w=960, h=544):
    rb = float(rb if rb is not None else REF_BOOST.get(scale, 2.55))
    rba = float(rba if rba is not None else REF_BOOST_A.get(scale, 1.00))
    return {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "krea2_turbo_int8_convrot.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": "krea2_identity_edit_v2_r64.safetensors", "strength_model": 1.0}},
        "3": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"}},
        "4": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_vae.safetensors"}},
        "5": {"class_type": "LoadImage", "inputs": {"image": scene_img}},
        "6": {"class_type": "LoadImage", "inputs": {"image": actor_img}},
        "7": {"class_type": "VAEEncode", "inputs": {"pixels": ["5", 0], "vae": ["4", 0]}},
        "8": {"class_type": "VAEEncode", "inputs": {"pixels": ["6", 0], "vae": ["4", 0]}},
        "9": {"class_type": "EmptyLatentImage", "inputs": {"width": w, "height": h, "batch_size": 1}},
        "10": {"class_type": "Krea2EditModelPatch", "inputs": {
            "model": ["2", 0], "source_latent": ["7", 0], "source_latent_b": ["8", 0],
            "ref_boost": rb, "ref_boost_a": rba, "fit_mode": "fit",
            "vae": ["4", 0], "source_image": ["5", 0], "source_image_b": ["6", 0],
            "target_latent": ["9", 0], "ref_position": "beside (right of target)", "ref_position_a": "aligned (trained)",
        }},
        "11": {"class_type": "Krea2EditGroundedEncode", "inputs": {
            "clip": ["3", 0], "prompt": prompt, "image": ["5", 0], "image_b": ["6", 0],
            "grounding_px": 768, "system_prompt": sys_prompt,
        }},
        "12": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["11", 0]}},
        "13": {"class_type": "KSampler", "inputs": {
            "model": ["10", 0], "positive": ["11", 0], "negative": ["12", 0], "latent_image": ["9", 0],
            "seed": seed, "steps": 8, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,
        }},
        "14": {"class_type": "VAEDecode", "inputs": {"samples": ["13", 0], "vae": ["4", 0]}},
        "15": {"class_type": "SaveImage", "inputs": {"images": ["14", 0], "filename_prefix": out_stem}},
    }

def run_krea2_img(wf, dst_path: Path):
    t0 = time.time()
    pid = _http_json(f"{SERVER}/prompt", {"prompt": wf, "client_id": str(uuid.uuid4())})["prompt_id"]
    while True:
        hist = _http_json(f"{SERVER}/history/{pid}", timeout=30)
        if pid in hist:
            st = hist[pid].get("status", {})
            if st.get("status_str") == "error":
                raise RuntimeError(str(st)[:500])
            for no in hist[pid].get("outputs", {}).values():
                for im in (no.get("images") or []):
                    qs = urllib.parse.urlencode({"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")})
                    with urllib.request.urlopen(f"{SERVER}/view?{qs}", timeout=60) as r:
                        dst_path.write_bytes(r.read())
                    return round(time.time() - t0, 1)
            raise RuntimeError("No image output")
        time.sleep(1.5)

print("Krea-2 Scale-Locked Swap helpers loaded! Use build_krea2_swap() + run_krea2_img() or launch a detached batch.")


In [ ]:
# CELL 4: Stage 2 — MiniMax-H3 Warm-RAM B2 Helpers (FP8_scaled + SageAttention + Kijai 8-Step + EasyCache 0.10)
# Automatically evicts Krea-2 via /free before loading H3 (prevents 52GB L4 RAM OOM!) & measures MAD jump spikes.
import json, shutil, subprocess, time, urllib.parse, urllib.request, uuid
import numpy as np
from pathlib import Path

SERVER = "http://127.0.0.1:8188"
COMFY_OUT = Path("/content/ComfyUI/output")
VID_DIR = Path("/content/mv_studio/videos")
VID_DIR.mkdir(parents=True, exist_ok=True)
UNET_FP8 = "minimax_h3_fl2va_pruned_fp8_scaled.safetensors"

def free_comfy():
    """Mandatory handoff before switching from Krea-2 (19GB) to MiniMax-H3 (44GB) on 52GB L4 RAM."""
    req = urllib.request.Request(
        f"{SERVER}/free",
        data=json.dumps({"unload_models": True, "free_memory": True}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=20):
            pass
        time.sleep(3)
    except Exception:
        pass

def build_h3_b2_workflow(first_img, prompt, frames=107, seed=55001, prefix="h3_clip", last_img=None, w=960, h=544, easycache=0.10):
    """Builds the winning L4/G4/A100 B2 workflow (FP8_scaled + Kijai 8-step v1.2 + optional EasyCache 0.10)."""
    wf = {
        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": UNET_FP8, "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "type": "minimax", "device": "default"}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"}},
        "4": {"class_type": "VAELoader", "inputs": {"vae_name": "minimax_h3_audio_vae_fp32.safetensors"}},
        "5": {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": "minimax_h3_fl2v_turbo_8step_v1.2_768p_pruned_avg_rank_24_bf16.safetensors", "strength_model": 1.0}},
        "7": {"class_type": "MiniMaxH3SigmaShift", "inputs": {"model": ["5", 0], "shift_video": 12.0, "shift_audio": 3.0}},
        "10": {"class_type": "LoadImage", "inputs": {"image": first_img}},
    }
    mref = ["7", 0]
    if easycache and float(easycache) > 0:
        wf["8"] = {"class_type": "EasyCache", "inputs": {"model": mref, "reuse_threshold": float(easycache), "start_percent": 0.25, "end_percent": 0.85, "verbose": False}}
        mref = ["8", 0]
    i2v_in = {"clip": ["2", 0], "vae": ["3", 0], "prompt": prompt, "width": w, "height": h, "length": int(frames), "first_frame": ["10", 0]}
    if last_img:
        wf["11"] = {"class_type": "LoadImage", "inputs": {"image": last_img}}
        i2v_in["last_frame"] = ["11", 0]
    wf["20"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": i2v_in}
    wf["21"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": int(seed)}}
    wf["22"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}}
    wf["23"] = {"class_type": "BasicScheduler", "inputs": {"model": mref, "scheduler": "simple", "steps": 8, "denoise": 1.0}}
    wf["24"] = {"class_type": "BasicGuider", "inputs": {"model": mref, "conditioning": ["20", 0]}}
    wf["25"] = {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["21", 0], "guider": ["24", 0], "sampler": ["22", 0], "sigmas": ["23", 0], "latent_image": ["20", 1]}}
    wf["26"] = {"class_type": "VAEDecode", "inputs": {"samples": ["25", 0], "vae": ["3", 0]}}
    wf["27"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["25", 0], "vae": ["4", 0]}}
    wf["28"] = {"class_type": "CreateVideo", "inputs": {"images": ["26", 0], "audio": ["27", 0], "fps": 24, "bit_depth": 8}}
    wf["29"] = {"class_type": "SaveVideo", "inputs": {"video": ["28", 0], "filename_prefix": prefix, "format": "auto", "codec": "auto"}}
    return wf

def measure_mad_spike(mp4_path):
    raw = subprocess.check_output(["ffmpeg", "-v", "error", "-i", str(mp4_path), "-vf", "scale=240:136", "-f", "rawvideo", "-pix_fmt", "rgb24", "-"])
    arr = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 136, 240, 3).astype(np.float32)
    diffs = np.mean(np.abs(arr[1:] - arr[:-1]), axis=(1, 2, 3))
    spikes = [diffs[i] / ((diffs[i-1] + diffs[i+1]) / 2.0 + 0.15) for i in range(1, len(diffs) - 1)]
    return round(float(diffs.max()), 2), round(float(np.median(diffs)), 2), round(float(max(spikes) if spikes else 1.0), 2)

print("MiniMax-H3 Warm-RAM B2 helpers ready! Call free_comfy() before your first H3 job after Krea-2.")


In [ ]:
# CELL 5: Stage 3 — Queue Any Validated Workflow JSON (Ref2VA Dialogue / Custom CLI JSON)
# Non-blocking submit: returns prompt_id immediately -> monitor progress in Cell 6!
import json, urllib.request, uuid
from pathlib import Path

WORKFLOW_JSON = ""  # e.g. "/content/mv_studio/inputs/scene01_wf.json" (built by h3_drama_cli.py build-workflow)

if WORKFLOW_JSON:
    raw = json.loads(Path(WORKFLOW_JSON).read_text(encoding="utf-8"))
    wf = raw.get("workflow", raw)
    req = urllib.request.Request(
        "http://127.0.0.1:8188/prompt",
        data=json.dumps({"prompt": wf, "client_id": str(uuid.uuid4())}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        res = json.loads(r.read().decode("utf-8"))
    print("QUEUED NON-BLOCKING -> prompt_id:", res.get("prompt_id"))
else:
    print("Set WORKFLOW_JSON to a validated JSON path from h3_drama_cli.py build-workflow and run.")


In [ ]:
# CELL 6: Live Studio Monitor (GPU / RAM / Live Cloudflared Tunnel URL / Active Logs)
import os, subprocess
from pathlib import Path

print("=== GPU & SYSTEM RAM ===")
print(subprocess.check_output(
    "nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader && free -h",
    shell=True, text=True
).strip())

tfile = Path("/tmp/tunnel_url.txt")
print("\n=== CLOUDFLARED TUNNEL ===")
print("URL:", tfile.read_text().strip() if tfile.exists() else "Not started (run Cell 1)")

for lp in ["/tmp/setup_studio_v3.log", "/tmp/batch_krea2.log", "/tmp/batch_h3.log", "/tmp/comfy_server.log"]:
    p = Path(lp)
    if p.exists():
        lines = p.read_text(errors="replace").splitlines()
        print(f"\n=== {lp} (last 12 / {len(lines)} lines) ===")
        print("\n".join(lines[-12:]))


In [ ]:
# CELL 7: CU SHIELD — Cleanly Stop ComfyUI & Unassign GPU Runtime Immediately (0 Idle CU Burn!)
# Run this cell as soon as all finished videos are downloaded/verified locally!
import os, time
os.system("pkill -9 -f 'main.py --listen' || true")
os.system("pkill -9 -f 'cloudflared' || true")
print("Stopped ComfyUI & cloudflared cleanly. Unassigning Colab GPU runtime to protect Compute Units...")
time.sleep(1)
from google.colab import runtime
runtime.unassign()
